In [28]:
!pip install polars

In [29]:
import polars as pl

In [30]:
df = pl.read_csv("/content/retail_store_sales.csv")

In [31]:
print(df.head)

<bound method DataFrame.head of shape: (12_575, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ Transacti ┆ Customer  ┆ Category  ┆ Item      ┆ … ┆ Payment   ┆ Location ┆ Transacti ┆ Discount  │
│ on ID     ┆ ID        ┆ ---       ┆ ---       ┆   ┆ Method    ┆ ---      ┆ on Date   ┆ Applied   │
│ ---       ┆ ---       ┆ str       ┆ str       ┆   ┆ ---       ┆ str      ┆ ---       ┆ ---       │
│ str       ┆ str       ┆           ┆           ┆   ┆ str       ┆          ┆ str       ┆ bool      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ TXN_68673 ┆ CUST_09   ┆ Patisseri ┆ Item_10_P ┆ … ┆ Digital   ┆ Online   ┆ 2024-04-0 ┆ true      │
│ 43        ┆           ┆ e         ┆ AT        ┆   ┆ Wallet    ┆          ┆ 8         ┆           │
│ TXN_37319 ┆ CUST_22   ┆ Milk      ┆ Item_17_M ┆ … ┆ Digital   ┆ Online   ┆ 2023-07-2 ┆ true      │
│ 86        ┆           ┆ Products  ┆ I

In [32]:
print(df.columns)

['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']


In [59]:
print(df.dtypes)

[String, String, String, String, Float64, Float64, Float64, String, String, String, Boolean]


In [33]:
print(df.shape)

(12575, 11)


In [34]:
print(df.select(['Price Per Unit','Quantity','Total Spent']).head())

shape: (5, 3)
┌────────────────┬──────────┬─────────────┐
│ Price Per Unit ┆ Quantity ┆ Total Spent │
│ ---            ┆ ---      ┆ ---         │
│ f64            ┆ f64      ┆ f64         │
╞════════════════╪══════════╪═════════════╡
│ 18.5           ┆ 10.0     ┆ 185.0       │
│ 29.0           ┆ 9.0      ┆ 261.0       │
│ 21.5           ┆ 2.0      ┆ 43.0        │
│ 27.5           ┆ 9.0      ┆ 247.5       │
│ 12.5           ┆ 7.0      ┆ 87.5        │
└────────────────┴──────────┴─────────────┘


In [35]:
df.null_count()

Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,1213,609,604,604,0,0,0,4199


In [49]:
df = df.with_columns(
  pl.when(pl.col("Price Per Unit").is_null())
  .then(
      pl.col("Total Spent")/pl.col("Quantity")
  )
  .otherwise(pl.col("Price Per Unit"))
  .alias("Price Per Unit")
 )

In [50]:
df = df.with_columns(
    pl.when(pl.col("Quantity").is_null())
    .then(
        pl.col("Total Spent") / pl.col("Price Per Unit")
    )
  .otherwise(pl.col("Quantity"))
  .alias("Quantity")
)

In [51]:
df = df.with_columns(
    pl.when(pl.col("Total Spent").is_null())
    .then(
        pl.col("Price Per Unit") * pl.col("Quantity")
    )
  .otherwise(pl.col("Total Spent"))
  .alias("Total Spent")
)

In [52]:
df.null_count()

Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,1213,0,604,604,0,0,0,4199


In [53]:
df = df.drop_nulls()

In [54]:
df.null_count()

Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0


In [55]:
print(df.shape)

(7579, 11)


In [58]:
df.write_csv("Store_sales_clean.csv")